# 🗂️ 공공데이터 API 실습 모음집
**대상:** 빅데이터 과정 수강생  
**목표:** 다양한 주제의 공공 API를 직접 찾고, 호출하고, 데이터를 분석한다

---
## 📌 공통 패턴 (모든 API에 동일하게 적용!)
```python
import requests

API_KEY = "본인키".strip()          # 1. 키 설정
url = "https://apis.data.go.kr/..." # 2. 주소 설정
params = {"serviceKey": API_KEY}    # 3. 파라미터 설정
response = requests.get(url, params=params)  # 4. 요청
data = response.json()              # 5. JSON 변환
```

---
## 📋 실습 목록
| 번호 | 주제 | API | 포털 검색어 |
|------|------|-----|------------|
| 1 | 🌤️ 날씨 | 기상청 단기예보 | `기상청 단기예보` |
| 2 | 🍽️ 음식/위생 | 식품안전나라 음식점 | `일반음식점` |
| 3 | 🏥 의료 | 전국 병원/약국 | `건강보험심사평가원 병원` |
| 4 | 🎭 문화/관광 | 한국관광공사 행사정보 | `한국관광공사 관광정보` |
| 5 | 🚗 교통 | 국가교통정보 주차장 | `주차장 정보` |
| 6 | 💊 의약품 | 의약품 정보 | `의약품개요정보` |

> 💡 **팁:** data.go.kr 검색창에 위 검색어를 입력하고 직접 찾아보세요!

In [1]:
import nest_asyncio
import os
from dotenv import load_dotenv

nest_asyncio.apply()

load_dotenv()

API_KEY = os.getenv('OPENAPI_API_KEY')
API_KEY_M = os.getenv('OPENAPI_KEY_MOVIE')

In [2]:
# ✅ 공통 설정 - 모든 실습에서 사용
import requests
import json
import pandas as pd
from datetime import datetime

def pretty(data):
    """JSON 보기 좋게 출력"""
    print(json.dumps(data, ensure_ascii=False, indent=2))

def check_response(response):
    """응답 상태 확인"""
    if response.status_code == 200:
        print("✅ 성공! (200)")
    elif response.status_code == 401:
        print("🔴 API 키 오류 (401) - 키를 확인하세요")
    elif response.status_code == 400:
        print("🔴 파라미터 오류 (400) - 필수값을 확인하세요")
    else:
        print(f"🔴 오류: {response.status_code}")
    return response.status_code == 200

print("공통 설정 완료! ✅")

공통 설정 완료! ✅


---
# 🌤️ 실습 1. 기상청 단기예보

**포털 검색:** `기상청 단기예보 조회서비스`  
**포털 URL:** https://www.data.go.kr/data/15084084/openapi.do

### 📝 API 문서 탐색 미션
```
□ 초단기실황조회 엔드포인트: getUltraSrtNcst
□ 필수 파라미터: base_date, base_time, nx, ny
□ nx, ny 란? → 기상청 격자 좌표 (위경도 아님!)
□ 대구광역시 좌표: nx=89, ny=90
```

### 날씨 카테고리 코드표
| 코드 | 항목 | 단위 |
|------|------|------|
| T1H | 기온 | ℃ |
| RN1 | 1시간 강수량 | mm |
| REH | 습도 | % |
| WSD | 풍속 | m/s |
| PTY | 강수형태 | 코드 |

> 강수형태: 0=없음, 1=비, 2=비/눈, 3=눈, 5=빗방울, 6=빗방울/눈날림, 7=눈날림

In [41]:
# 🌤️ 기상청 초단기실황 - 현재 날씨
now = datetime.now()
base_date = now.strftime("%Y%m%d")
base_time = now.strftime("%H00")  # 정시 기준

url_weather = "https://apis.data.go.kr/1360000/VilageFcstInfoService_2.0/getUltraSrtNcst"

params_weather = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "dataType": "JSON",
    "base_date": base_date,
    "base_time": base_time,
    "nx": "89",   # 대구광역시
    "ny": "90"
}

print(f"요청 날짜: {base_date}, 시간: {base_time}")
response_w = requests.get(url_weather, params=params_weather)

if check_response(response_w):
    data_w = response_w.json()
    items_w = data_w['response']['body']['items']['item']
    print(f"\nitem 리스트의 첫 번째 요소 : {items_w[0]}")

    print("\n📊 현재 날씨 데이터:")
    category_names = {
        "T1H": "기온(℃)", "RN1": "강수량(mm)",
        "REH": "습도(%)", "WSD": "풍속(m/s)", "PTY": "강수형태"
    }
    for item in items_w:
        name = category_names.get(item['category'], item['category'])
        print(f"    {name}: {item['obsrValue']}")

요청 날짜: 20260604, 시간: 1400
✅ 성공! (200)

item 리스트의 첫 번째 요소 : {'baseDate': '20260604', 'baseTime': '1400', 'category': 'PTY', 'nx': 89, 'ny': 90, 'obsrValue': '0'}

📊 현재 날씨 데이터:
    강수형태: 0
    습도(%): 49
    강수량(mm): 0
    기온(℃): 28.5
    UUU: -0.8
    VEC: 158
    VVV: 2.2
    풍속(m/s): 2.3


In [44]:
# 🎯 미션: 내가 원하는 도시 날씨 조회
# nx, ny 좌표표: https://www.weather.go.kr/w/observation/land/grid.do 에서 확인

# 주요 도시 좌표
cities = {
    "서울": (60, 127),
    "부산": (98, 76),
    "대구": (89, 90),
    "인천": (55, 124),
    "광주": (58, 74),
    "대전": (67, 100)
}

# 여기서 도시를 선택하세요!
my_city = "대전"  
nx, ny = cities[my_city]

print(f"선택한 도시: {my_city} (nx={nx}, ny={ny})")

params_weather = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "dataType": "JSON",
    "base_date": base_date,
    "base_time": base_time,
    "nx": str(nx),   
    "ny": str(ny)
}

print(f"요청 날짜: {base_date}, 시간: {base_time}")
response_wb = requests.get(url_weather, params=params_weather)

if check_response(response_wb):
    data_wb = response_wb.json()
    items_wb = data_wb['response']['body']['items']['item']

    print("\n📊 현재 날씨 데이터:")
    for item in items_wb:
        name = category_names.get(item['category'], item['category'])
        print(f"    {name}: {item['obsrValue']}")

선택한 도시: 대전 (nx=67, ny=100)
요청 날짜: 20260604, 시간: 1400
✅ 성공! (200)

📊 현재 날씨 데이터:
    강수형태: 0
    습도(%): 54
    강수량(mm): 0
    기온(℃): 27.8
    UUU: 1.4
    VEC: 196
    VVV: 4.9
    풍속(m/s): 5


In [45]:
data_wb

{'response': {'header': {'resultCode': '00', 'resultMsg': 'NORMAL_SERVICE'},
  'body': {'dataType': 'JSON',
   'items': {'item': [{'baseDate': '20260604',
      'baseTime': '1400',
      'category': 'PTY',
      'nx': 67,
      'ny': 100,
      'obsrValue': '0'},
     {'baseDate': '20260604',
      'baseTime': '1400',
      'category': 'REH',
      'nx': 67,
      'ny': 100,
      'obsrValue': '54'},
     {'baseDate': '20260604',
      'baseTime': '1400',
      'category': 'RN1',
      'nx': 67,
      'ny': 100,
      'obsrValue': '0'},
     {'baseDate': '20260604',
      'baseTime': '1400',
      'category': 'T1H',
      'nx': 67,
      'ny': 100,
      'obsrValue': '27.8'},
     {'baseDate': '20260604',
      'baseTime': '1400',
      'category': 'UUU',
      'nx': 67,
      'ny': 100,
      'obsrValue': '1.4'},
     {'baseDate': '20260604',
      'baseTime': '1400',
      'category': 'VEC',
      'nx': 67,
      'ny': 100,
      'obsrValue': '196'},
     {'baseDate': '20260604',
   

---
# 🍽️ 실습 2. 전국 일반음식점 정보

**포털 검색:** `일반음식점`  
**포털 URL:** https://www.data.go.kr/data/15096283/standard.do

> 이 API는 각 지자체에서 제공하므로 **지역별로 별도 신청** 필요!  
> 대구시: `대구 일반음식점` 으로 검색

### 📝 탐색 미션
```
□ 어떤 필드들이 있나요? (업소명, 주소, 업종, ...)
□ 필수 파라미터는?
□ 페이징 파라미터는? (numOfRows, pageNo)
```

In [35]:
# 🍽️ 일반음식점 정보 조회
# ※ 지역마다 URL이 다릅니다! Swagger에서 확인한 URL을 입력하세요

url_food = "https://apis.data.go.kr/6260000/BusanTblFnrstrnStusService/getTblFnrstrnStusInfo"

params_food = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "resultType":"json"
}

response_f = requests.get(url_food, params=params_food)
print("응답 코드:", response_f.status_code)

if response_f.status_code == 200:
    data_f = response_f.json()
    pretty(data_f)

응답 코드: 200
{
  "response": {
    "header": {
      "resultMsg": "NORMAL_CODE",
      "resultCode": "00"
    },
    "body": {
      "totalCount": "268",
      "items": {
        "item": [
          {
            "bsnsSector": "일반음식점",
            "bsnsCond": "식육(숯불구이)",
            "bsnsNm": "청송양곱창",
            "addrRoad": "부산광역시 수영구 남천동로108번길 8, 1~2층 (남천동)",
            "addrJibun": "부산광역시 수영구 남천동 6 - 13",
            "menu": "양곱창",
            "tel": "051-627-5291",
            "specDate": "2014-11-03",
            "ovrdDate": "2023-11-01",
            "gugun": "부산광역시 수영구",
            "dataDay": "2023-12-31",
            "lat": "35.1457746",
            "lng": "129.1120335"
          },
          {
            "bsnsSector": "일반음식점",
            "bsnsCond": "회집",
            "bsnsNm": "포항물횟집",
            "addrRoad": "부산광역시 수영구 광안해변로290번길 17, 1~3층 (민락동)",
            "addrJibun": "부산광역시 수영구 민락동 181 - 120 1~3층",
            "menu": "생선회, 물회",
            "tel": "051-752-5732",
       

In [39]:
# 🎯 미션: 음식점 데이터를 pandas DataFrame으로 만들기
# (데이터 구조를 먼저 확인한 후 필드명에 맞게 수정하세요)

# 예시 - 응답 구조에 맞게 수정 필요
items = data_f["response"]["body"]["items"]["item"]
df = pd.DataFrame(items)
print(df.head())
print("\n컬럼 목록:", df.columns.tolist())

# print("↑ 주석을 해제하고 실행해보세요!")

  bsnsSector  bsnsCond  bsnsNm                             addrRoad  \
0      일반음식점  식육(숯불구이)   청송양곱창    부산광역시 수영구 남천동로108번길 8, 1~2층 (남천동)   
1      일반음식점        회집   포항물횟집  부산광역시 수영구 광안해변로290번길 17, 1~3층 (민락동)   
2      일반음식점        회집   청산도횟집            부산광역시 동래구 명륜로 80-10 (수안동)   
3      일반음식점        한식  논두렁추어탕        부산광역시 동래구 명륜로139번길 42-5 (명륜동)   
4      일반음식점        한식     대궐집          부산광역시 동래구 동래로136번길 26 (복천동)   

                      addrJibun     menu           tel    specDate  \
0          부산광역시 수영구 남천동 6 - 13      양곱창  051-627-5291  2014-11-03   
1  부산광역시 수영구 민락동 181 - 120 1~3층  생선회, 물회  051-752-5732  2010-06-30   
2           부산광역시 동래구 수안동 191-7      생선회  051-552-9528  2012-11-06   
3          부산광역시 동래구 명륜동 551-13      추어탕  051-553-5769  2012-11-06   
4           부산광역시 동래구 복천동 319-3      소갈비  051-552-0127  2002-05-30   

     ovrdDate      gugun     dataDay          lat          lng  
0  2023-11-01  부산광역시 수영구  2023-12-31   35.1457746  129.1120335  
1  2023-11-01  부산광역시 수

In [41]:
df2 = df.loc[:,['bsnsSector', 'bsnsCond', 'bsnsNm', 'menu', 'gugun', 'tel']]
df2

,bsnsSector,bsnsCond,bsnsNm,menu,gugun,tel
0,일반음식점,식육(숯불구이),청송양곱창,양곱창,부산광역시 수영구,051-627-5291
1,일반음식점,회집,포항물횟집,"생선회, 물회",부산광역시 수영구,051-752-5732
2,일반음식점,회집,청산도횟집,생선회,부산광역시 동래구,051-552-9528
3,일반음식점,한식,논두렁추어탕,추어탕,부산광역시 동래구,051-553-5769
4,일반음식점,한식,대궐집,소갈비,부산광역시 동래구,051-552-0127
5,일반음식점,한식,대하찹쌀아구찜,아구찜,부산광역시 동래구,051-505-8862
6,일반음식점,분식,마루한,만두국,부산광역시 동래구,051-553-3070
7,일반음식점,탕류(보신용),맛나감자탕사직점,감자탕,부산광역시 동래구,051-503-3630
8,일반음식점,한식,명가,갈비,부산광역시 동래구,051-553-5989
9,일반음식점,한식,구기영 조방낙지,"낙지전골,낙지볶음",부산광역시 동래구,051-528-7055


In [45]:
df2 = df2.rename(columns={'bsnsSector':'업종',
                    'bsnsCond':'업태',
                    'bsnsNm':'업소명',
                    'menu':'메뉴',
                    'gugun':'구군명',
                    'tel':'전화번호'
                    })
df2

,업종,업태,업소명,메뉴,구군명,전화번호
0,일반음식점,식육(숯불구이),청송양곱창,양곱창,부산광역시 수영구,051-627-5291
1,일반음식점,회집,포항물횟집,"생선회, 물회",부산광역시 수영구,051-752-5732
2,일반음식점,회집,청산도횟집,생선회,부산광역시 동래구,051-552-9528
3,일반음식점,한식,논두렁추어탕,추어탕,부산광역시 동래구,051-553-5769
4,일반음식점,한식,대궐집,소갈비,부산광역시 동래구,051-552-0127
5,일반음식점,한식,대하찹쌀아구찜,아구찜,부산광역시 동래구,051-505-8862
6,일반음식점,분식,마루한,만두국,부산광역시 동래구,051-553-3070
7,일반음식점,탕류(보신용),맛나감자탕사직점,감자탕,부산광역시 동래구,051-503-3630
8,일반음식점,한식,명가,갈비,부산광역시 동래구,051-553-5989
9,일반음식점,한식,구기영 조방낙지,"낙지전골,낙지볶음",부산광역시 동래구,051-528-7055


---
# 🏥 실습 3. 전국 병원/약국 정보

**포털 검색:** `건강보험심사평가원 병원정보서비스`  
**포털 URL:** https://www.data.go.kr/data/15001698/openapi.do

### 📝 탐색 미션
```
□ 병원 조회 엔드포인트: getBsshInfoInqire
□ 약국 조회 엔드포인트: getParmacyBasisList
□ 지역코드: 110000 (서울), 210000 (부산), 230000 (대구), 220000 (인천)
```

In [3]:
# 🏥 대구 지역 병원 정보 조회
url_hospital = "https://apis.data.go.kr/B551182/hospInfoServicev2/getHospBasisList"

params_hospital = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "sidoCd": "230000",  # 대구광역시 코드
    "_type": "json"
}

response_h = requests.get(url_hospital, params=params_hospital)
print("응답 코드:", response_h.status_code)

if response_h.status_code == 200:
    data_h = response_h.json()
    items = data_h["response"]["body"]["items"]["item"]
    
    print(f"\n총 {data_h['response']['body']['totalCount']}개 병원")
    print("\n병원 목록 (10개):")
    for h in items:
        print(f"  🏥 {h['yadmNm']} | {h['clCdNm']} | {h['addr']}")

응답 코드: 200

총 4228개 병원

병원 목록 (10개):
  🏥 경북대학교병원 | 상급종합 | 대구광역시 중구 동덕로 130, (삼덕동2가, 경북대학교병원)
  🏥 계명대학교동산병원 | 상급종합 | 대구광역시 달서구 달구벌대로 1035, (신당동)
  🏥 대구가톨릭대학교병원 | 상급종합 | 대구광역시 남구 두류공원로17길 33, (대명동)
  🏥 영남대학교병원 | 상급종합 | 대구광역시 남구 현충로 170, (대명동)
  🏥 칠곡경북대학교병원 | 상급종합 | 대구광역시 북구 호국로 807, 칠곡경북대학교병원(임상실습동 1~11층 포함) (학정동)
  🏥 (재)미리내천주성삼성직수도회천주성삼병원 | 종합병원 | 대구광역시 수성구 달구벌대로 3190, (신매동, 천주성삼병원)
  🏥 강남종합병원 | 종합병원 | 대구광역시 동구 동촌로 207, 강남병원 (방촌동)
  🏥 계명대학교대구동산병원 | 종합병원 | 대구광역시 중구 달성로 56, 계명대학교대구동산병원 (동산동)
  🏥 곽병원 | 종합병원 | 대구광역시 중구 국채보상로 531, (수동)
  🏥 나사렛종합병원 | 종합병원 | 대구광역시 달서구 월배로 97, 나사렛병원 (진천동)


In [ ]:
# 시도 코드 찾기

# for h in items:
#     if '인천' in h['addr']:
#         print(f"{h['sidoCd']} | {h['yadmNm']} | {h['addr']}")

220000 | 가톨릭대학교인천성모병원 | 인천광역시 부평구 동수로 56, (부평동)
220000 | 의료법인 길의료재단 길병원 | 인천광역시 남동구 남동대로774번길 21-0, 의료법인길의료재단길병원
220000 | 인하대학교의과대학부속병원 | 인천광역시 중구 인항로 27-0, 인하대학병원
220000 | (의)나사렛의료재단 나사렛국제병원 | 인천광역시 연수구 먼우금로 98,  (동춘동)
220000 | (의)성세의료재단 뉴성민병원 | 인천광역시 서구 신석로 70, (석남동)


In [51]:
# 💊 약국 정보 조회
url_pharmacy = "https://apis.data.go.kr/B551182/pharmacyInfoService/getParmacyBasisList"

params_ph = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "sidoCd": "230000",  # 대구
    "_type": "json"
}

response_ph = requests.get(url_pharmacy, params=params_ph)

if check_response(response_ph):
    data_ph = response_ph.json()
    items_ph = data_ph["response"]["body"]["items"]["item"]
    
    print("\n약국 목록:")
    for p in items_ph:
        print(f"  💊 {p['yadmNm']} | {p['addr']}")

✅ 성공! (200)

약국 목록:
  💊 영남약국 | 대구광역시 남구 중앙대로29길 33, (대명동)
  💊 동구시장심야약국 | 대구광역시 동구 화랑로 123, 1층 115호 (효목동, 유림아파트)
  💊 소나무약국 | 대구광역시 중구 명덕로 241, 303동 B107호 (대봉동, 대봉 서한이다음)
  💊 봄약국 | 대구광역시 동구 아양로 162, 1층 (신암동)
  💊 백만약국 | 대구광역시 동구 해동로 39, 1층 (지저동)
  💊 다나은약국 | 대구광역시 달서구 월배로 196, 1층 (상인동)
  💊 동화약국 | 대구광역시 달서구 상인서로 94, 208호 (상인동, 동화태왕한양상가)
  💊 대우약국 | 대구광역시 달성군 논공읍 논공로 210, (논공읍)
  💊 열린약국 | 대구광역시 수성구 지범로 113, (지산동)
  💊 대학일번약국 | 대구광역시 남구 두류공원로19길 31, 2층 (대명동)


In [12]:
# 🎯 미션: 내 주변 특정 진료과목 병원 찾기
# 진료과목 코드 예시: 01=내과, 02=신경과, 03=정신건강의학과, 05=정형외과,
#                   08=성형외과, 10=산부인과, 12=안과, 13=이비인후과, 14=피부과

# params에 dgsbjtCd 파라미터를 추가해보세요
# params_hospital["dgsbjtCd"] = target_dept

# 여기에 코드를 작성해보세요!
target_dept = "08"  # ← 원하는 진료과목 코드로 바꾸세요

params_hospital = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "sidoCd": "230000",  # 대구광역시 코드
    "_type": "json",
    "dgsbjtCd": target_dept
}

response_hp = requests.get(url_hospital, params=params_hospital)
data_hp = response_hp.json()
items_hp = data_hp["response"]["body"]["items"]["item"]

print('<대구 성형외과 진료 병원 목록>')
for item in items_hp:
    print(f"  🏥 {item['yadmNm']} | {item['addr']} | {item['telno']}")

<대구 성형외과 진료 병원 목록>
  🏥 경북대학교병원 | 대구광역시 중구 동덕로 130, (삼덕동2가, 경북대학교병원) | 053-200-5114
  🏥 계명대학교동산병원 | 대구광역시 달서구 달구벌대로 1035, (신당동) | 1577-6622
  🏥 대구가톨릭대학교병원 | 대구광역시 남구 두류공원로17길 33, (대명동) | 053-650-3000
  🏥 영남대학교병원 | 대구광역시 남구 현충로 170, (대명동) | 053-623-8001
  🏥 칠곡경북대학교병원 | 대구광역시 북구 호국로 807, 칠곡경북대학교병원(임상실습동 1~11층 포함) (학정동) | 053-200-2000
  🏥 대구파티마병원 | 대구광역시 동구 아양로 99, (신암동) | 053-1688-7770
  🏥 더블유병원 | 대구광역시 달서구 달구벌대로 1632, 더블유병원 (감삼동) | 053-550-5000
  🏥 삼일병원 | 대구광역시 달서구 월배로 436, 지하1,지상1~11층 (송현동) | 053-659-3100
  🏥 MS재건병원 | 대구광역시 중구 동덕로 194, (동인동1가, 동화빌딩) | 053-653-0119
  🏥 광개토병원 | 대구광역시 중구 중앙대로 366, 지하1,1,2,3,5,7,11,12,14,17층 (덕산동) | 053-565-1190


---
# 🎭 실습 4. 한국관광공사 문화/행사 정보

**포털 검색:** `한국관광공사 관광정보 서비스`  
**포털 URL:** https://www.data.go.kr/data/15101578/openapi.do

> ⚠️ 이 API는 **별도 사이트에서도 신청 가능!**  
> https://api.visitkorea.or.kr/ 에서 회원가입 후 키 발급

### 📝 탐색 미션
```
□ 행사정보 엔드포인트: searchFestival
□ 지역기반 관광정보: areaBasedList
□ 지역코드: 1=서울, 2=인천, 4=대구, 5=광주, 6=부산
□ 콘텐츠타입: 12=관광지, 14=문화시설, 15=행사/공연/축제
```

In [13]:
# 🎭 대구 지역 행사/축제 정보
url_tour = "https://apis.data.go.kr/B551011/KorService2/searchFestival2"

params_tour = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "MobileOS": "ETC",
    "MobileApp": "TestApp",
    "_type": "json",
    "areaCode": "6",       # 부산
    "eventStartDate": datetime.now().strftime("%Y%m%d") # 오늘 이후 행사 
}

response_t = requests.get(url_tour, params=params_tour)

if check_response(response_t):
    data_t = response_t.json()

# data_t
    items_t = data_t["response"]["body"]["items"]["item"]
    
    print("\n🎉 부산 행사/축제 목록:")
    for event in items_t:
        print(f"  🎭 {event['title']}")
        print(f"     기간: {event.get('eventstartdate','')} ~ {event.get('eventenddate','')}")
        print(f"     장소: {event.get('addr1','')}")

✅ 성공! (200)

🎉 부산 행사/축제 목록:
  🎭 광안리 M(Marvelous) 드론 라이트쇼
     기간: 20260101 ~ 20261231
     장소: 부산광역시 수영구 광안해변로 219 (광안동)


In [ ]:
# 🎯 미션: 지역 관광지 정보 조회
url_area = "https://apis.data.go.kr/B551011/KorService2/areaBasedList2"

params_area = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "MobileOS": "ETC",
    "MobileApp": "TestApp",
    "_type": "json",
    "areaCode": "4",        # 대구
    "contentTypeId": "12"   # 12=관광지, 14=문화시설, 15=축제행사
}

response_a = requests.get(url_area, params=params_area)

if check_response(response_a):
    data_a = response_a.json()
    spots = data_a["response"]["body"]["items"]["item"]
    
    print("\n🗺️ 대구 관광지 목록:")
    for spot in spots:
        print(f"  📍 {spot['title']} | {spot.get('addr1','')}")

✅ 성공! (200)

🗺️ 대구 관광지 목록:
  📍 가창 찐빵 골목 | 대구광역시 달성군 가창면 가창로 1099
  📍 감삼못공원 | 대구광역시 서구 서대구로3길 43 (내당동)
  📍 강정고령보 | 대구광역시 달성군 다사읍 강정본길 57
  📍 경극고택 | 대구광역시 동구 옻골로 195-3
  📍 경상감영공원 | 대구광역시 중구 경상감영길 99
  📍 고산골 메타쉐콰이어길 | 대구광역시 남구 용두2길 43
  📍 고산서당 | 대구광역시 수성구 성동로37길 39-3
  📍 공룡공원 | 대구광역시 남구 용두2길 43
  📍 관암사(대구) | 대구광역시 동구 갓바위로 350 (능성동)
  📍 관음사(대구) | 대구광역시 동구 둔산로 535


---
# 🚗 실습 5. 주차장 정보

**포털 검색:** `주차장 정보`  
**대구 주차장:** `대구 주차장`으로 검색

### 📝 탐색 미션
```
□ API 이름: 대구광역시 수성구 주차장 정보 조회 서비스
□ URL: https://apis.data.go.kr/3460000/ParkingLotInfoService
□ 필수 파라미터: serviceKey, numOfRows, pageNo
□ 기본 주차 시간(분 단위): parkingChrgeBsTime
□ 기본 주차 요금(원): parkingChrgeBsChrge
```

In [6]:
# 🚗 주차장 정보 조회
# ※ API 문서를 보고 직접 URL과 파라미터를 채워보세요!

url_park = "https://apis.data.go.kr/3460000/ParkingLotInfoService/parkingLotInfo"  # ← API 문서에서 찾은 URL 입력

params_park = {
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1"
    # ← 필수 파라미터 추가
}

if url_park:
    response_p = requests.get(url_park, params=params_park)
    print("응답 코드:", response_p.status_code)
    if response_p.status_code == 200:
        data_p = response_p.json()
        pretty(data_p)

응답 코드: 200
{
  "response": {
    "header": {
      "resultCode": "00",
      "resultMsg": "NORMAL_SERVICE"
    },
    "body": {
      "items": {
        "item": [
          {
            "prkPlceId": "155-2-000065",
            "prkPlceNm": "두산레포츠센터주차장",
            "prkPlceKnd": "2",
            "locplcAddr": "대구광역시 수성구 두산동 산 7-2",
            "prkPlceEntrcNm": "대구광역시 수성구 두산동 산 7-2",
            "prkPlceEntrcLa": "35.823076",
            "prkPlceEntrcLo": "128.621909",
            "parkingUs": "9",
            "arLevelSe": "1",
            "sysNe": "주차정보공유 시스템",
            "prkCmprtCo": "95",
            "cmptInstNm": "대구광역시청",
            "noDrvDayOpertnAt": "0",
            "sunOpertnStartTime": "08:00:00",
            "sunOpertnEndTime": "21:00:00",
            "monOpertnStartTime": "08:00:00",
            "monOpertnEndTime": "21:00:00",
            "tuesOpertnStartTime": "08:00:00",
            "tuesOpertnEndTime": "21:00:00",
            "wedOpertnStartTime": "08:00:00",
       

In [7]:
print(f'수성구 주차장 총 개수: {data_p["response"]["body"]["totalCount"]}개')

items_p = data_p["response"]["body"]["items"]["item"]

print('\n<주차장 목록>')
for item in items_p:
    print(f' 🅿 {item["prkPlceNm"]} | {item["parkngCo"]}대 | 기본 {item["parkingChrgeBsTime"]}분 | {item["parkingChrgeBsChrge"]}원')

수성구 주차장 총 개수: 329개

<주차장 목록>
 🅿 두산레포츠센터주차장 | 95대 | 기본 120분 | 2000원
 🅿 수성못공영주차장 | 105대 | 기본 30분 | 600원
 🅿 울루루광장공영주차장 | 67대 | 기본 30분 | 600원
 🅿 지산한라공영주차장 | 48대 | 기본 30분 | 400원
 🅿 범어공영주차장 | 62대 | 기본 30분 | 600원
 🅿 수성구청 주차장 | 138대 | 기본 30분 | 600원
 🅿 고산노인복지관 | 53대 | 기본 30분 | 400원
 🅿 대구여고북편공영주차장 | 71대 | 기본 30분 | 600원
 🅿 범어천로 공영1주차장 | 78대 | 기본 30분 | 600원
 🅿 범어천로 공영2주차장 | 80대 | 기본 30분 | 600원


---
# 💊 실습 6. 의약품 정보

**포털 검색:** `의약품개요정보`  
**포털 URL:** https://www.data.go.kr/data/15075057/openapi.do

### 📝 탐색 미션
```
□ 제품명으로 검색하는 파라미터: itemName
□ 업체명으로 검색하는 파라미터: entpName
□ 주요 응답 필드: itemName, entpName, efcyQesitm(효능), useMethodQesitm(용법)
```

In [19]:
# 💊 의약품 정보 조회
# url_drug = "https://apis.data.go.kr/1471000/DrugPrdtPrmsnInfoService04/getDrugPrdtPrmsnDtlInq04"
url_drug = "https://apis.data.go.kr/1471000/DrbEasyDrugInfoService/getDrbEasyDrugList"

params_drug = {
    "serviceKey": API_KEY,
    "numOfRows": "5",
    "pageNo": "1",
    "type": "json",
    "itemName": "타이레놀"  # ← 검색할 약품명
}

response_d = requests.get(url_drug, params=params_drug)

if check_response(response_d):
    data_d = response_d.json()
    items_d = data_d["body"]["items"]
    
    print()
    for drug in items_d:
        print(f"💊 {drug['itemName']}")
        print(f"   제조사: {drug.get('entpName','')}")
        print(f"   효능: {drug.get('efcyQesitm','')[:80]}...")  
        print()

✅ 성공! (200)

💊 어린이타이레놀산160밀리그램(아세트아미노펜)
   제조사: 켄뷰코리아판매유한회사
   효능: 이 약은 감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증), 치통, 관절통, 류마티양 동통(통증)에 사용합...

💊 타이레놀정500밀리그람(아세트아미노펜)
   제조사: 켄뷰코리아판매유한회사
   효능: 이 약은 감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증), 치통, 관절통, 류마티양 동통(통증)에 사용합...

💊 타이레놀콜드-에스정
   제조사: 켄뷰코리아판매유한회사
   효능: 이 약은 감기의 제증상(여러증상)(콧물, 코막힘, 재채기, 인후(목구멍)통, 기침, 오한(춥고 떨리는 증상), 발열, 두통, 관절통, 근육통)의...

💊 타이레놀8시간이알서방정(아세트아미노펜)
   제조사: 켄뷰코리아판매유한회사
   효능: 이 약은 해열 및 감기에 의한 동통(통증)과 두통, 치통, 근육통, 허리통증, 생리통, 관절통의 완화에 사용합니다.
...

💊 어린이타이레놀현탁액(아세트아미노펜)
   제조사: 켄뷰코리아판매유한회사
   효능: 이 약은 감기로 인한 발열 및 동통(통증), 두통, 신경통, 근육통, 월경통, 염좌통(삔 통증), 치통, 관절통, 류마티양 동통(통증)에 사용합...



In [30]:
# 🎯 미션: 내가 알고 있는 약 검색해보기
my_drug = "탁센"  # ← 원하는 약 이름으로 바꿔보세요
params_drug["itemName"] = my_drug

response_d2 = requests.get(url_drug, params=params_drug)
if check_response(response_d2):
    data_d2 = response_d2.json()
    print(f"'{my_drug}' 검색 결과:")
    pretty(data_d2)

✅ 성공! (200)
'탁센' 검색 결과:
{
  "header": {
    "resultCode": "00",
    "resultMsg": "NORMAL SERVICE."
  },
  "body": {
    "pageNo": 1,
    "totalCount": 7,
    "numOfRows": 5,
    "items": [
      {
        "entpName": "(주)녹십자",
        "itemName": "탁센연질캡슐(나프록센)",
        "itemSeq": "200710782",
        "efcyQesitm": "이 약은 류마티양 관절염, 골관절염(퇴행성 관절질환), 강직성 척추염, 건염(힘줄염), 급성통풍, 월경곤란증과 활액(윤활)낭염, 골격근장애(염좌(삠), 좌상(타박상), 외상(상처), 요천통(허리통증), 수술 후 동통(통증), 편두통, 발치(이를 뽑음)후 동통(통증)에 사용합니다.\n",
        "useMethodQesitm": "류마티양 관절염, 골관절염, 강직성 척추염에 성인은 1회 1~2캡슐(250~500 mg)씩, 1일 2회 12시간마다 복용합니다.\n\n급성통풍에 성인은 처음 복용량으로 3캡슐(750 mg)을 복용하고 발작이 없어질 때까지 8시간 간격으로 1캡슐(250 mg)씩 복용합니다.\n\n골격근장애, 수술후 동통(통증), 발치(이를 뽑음)후 동통(통증), 월경곤란증, 건염(힘줄염), 활액(윤활)낭염에 성인은 처음 복용량으로 2캡슐(500 mg)을 복용하고 6~8시간 간격으로 1캡슐(250 mg)씩 복용합니다.\n\n편두통에 성인은 처음 복용량으로 3캡슐(750 mg)을 복용합니다. 필요 시, 1일 1~2캡슐(250~500 mg)을 더 복용할 수 있으며 처음 복용량의 복용 30분 후에 복용합니다.\n\n1일 총용량이 5캡슐(1,250 mg)을 초과하지 않도록 합니다.\n\n연령, 증상에 따라 적절히 증감할 수 있습니다.\n",
        "atpnWarnQesitm": "매일 세

In [31]:
items_d2 = data_d2["body"]["items"]

print(f"<{my_drug} 약품 조회>\n")
for item in items_d2:
    print(f" 💊 {item['itemName']}")
    print(f"    효능 : {item['efcyQesitm'][:50]}...")
    print(f"    용법 : {item['useMethodQesitm'][:50]}...")
    print()

<탁센 약품 조회>

 💊 탁센연질캡슐(나프록센)
    효능 : 이 약은 류마티양 관절염, 골관절염(퇴행성 관절질환), 강직성 척추염, 건염(힘줄염), 급...
    용법 : 류마티양 관절염, 골관절염, 강직성 척추염에 성인은 1회 1~2캡슐(250~500 mg)씩...

 💊 탁센400이부프로펜연질캡슐
    효능 : 이 약은 감기로 인한 발열 및 동통(통증), 요(허리)통, 생리통, 류마티양 관절염, 연소...
    용법 : 류마티양 관절염, 골관절염, 강직성 척추염, 연조직손상, 비관절 류마티스질환, 급성통풍, ...

 💊 탁센이브연질캡슐
    효능 : 이 약은 두통, 치통, 발치(이를 뽑음)후 동통(통증), 인후(목구멍)통, 귀의 통증, 관...
    용법 : 만 15세 이상 또는 성인은 1회 1~2캡슐씩, 1일 1~3회, 4시간 이상 간격, 만 1...

 💊 탁센엠지연질캡슐
    효능 : 이 약은 두통, 치통, 발치후 동통(통증), 인후(목구멍)통, 귀의 통증, 관절통, 신경통...
    용법 : 만 15세 이상 및 성인은 1회 1~2캡슐, 1일 1~3회(복용간격은 4시간 이상), 만 ...

 💊 탁센덱시연질캡슐(덱시부프로펜)
    효능 : 이 약은 만성 다발성 관절염, 류마티스관절염, 관절증, 강직척추염, 외상 및 수술 후 통증...
    용법 : 성인은 1회 300 mg을 1일 2～4회 경구투여하십시오. 단, 1일 1,200 mg을 초...



---
# 🏆 자유 실습 - 내가 찾은 API

data.go.kr에서 **내가 관심 있는 데이터**를 직접 찾아서 실습해보세요!

### 탐색 → 실습 체크리스트
```
□ 1. data.go.kr에서 관심 키워드 검색
□ 2. 오픈 API 목록에서 원하는 것 선택
□ 3. 활용 신청 클릭
□ 4. Swagger UI에서 Try it out → 200 성공 확인
□ 5. 아래 셀에 파이썬 코드로 구현
□ 6. 응답 데이터 구조 파악 후 원하는 값 추출
```

### 💡 아이디어
- 🎬 영화진흥위원회 박스오피스 순위
- 🚇 지하철 실시간 위치
- 📚 도서관 장서 검색
- ⚽ 한국 축구 경기 결과
- 🌊 해수욕장 정보
- 🏠 부동산 실거래가

In [3]:
# 🏆 내가 찾은 API 실습

# API 이름: 주간/주말 박스오피스
# 포털 URL: https://www.kobis.or.kr/kobisopenapi/homepg/apiservice/searchServiceInfo.do
# 내가 뽑고 싶은 데이터: 주간 박스오피스 순위

my_url = "https://www.kobis.or.kr/kobisopenapi/webservice/rest/boxoffice/searchWeeklyBoxOfficeList.json"       # ← 내 API URL
my_params = {
    "key": API_KEY_M,
    "targetDt": "20260315",
    "itemPerPage": "10",
    "weekGb": "0" # 주간:0 / 주말:1 / 주중:2
    # "repNationCd": "K" # 한국:K / 외국:F
}

if my_url:
    my_response = requests.get(my_url, params=my_params)
    print("응답 코드:", my_response.status_code)
    if my_response.status_code == 200:
        data_m = my_response.json()
        pretty(data_m)
# else:
#     print("⚠️ URL을 먼저 입력하세요!")

응답 코드: 200
{
  "boxOfficeResult": {
    "boxofficeType": "주간 박스오피스",
    "showRange": "20260309~20260315",
    "yearWeekTime": "202611",
    "weeklyBoxOfficeList": [
      {
        "rnum": "1",
        "rank": "1",
        "rankInten": "0",
        "rankOldAndNew": "OLD",
        "movieCd": "20242837",
        "movieNm": "왕과 사는 남자",
        "openDt": "2026-02-04",
        "salesAmt": "18996318830",
        "salesShare": "77.2",
        "salesInten": "-10260112660",
        "salesChange": "-35.1",
        "salesAcc": "130008120510",
        "audiCnt": "1964055",
        "audiInten": "-1055218",
        "audiChange": "-34.9",
        "audiAcc": "13467690",
        "scrnCnt": "2262",
        "showCnt": "59789"
      },
      {
        "rnum": "2",
        "rank": "2",
        "rankInten": "0",
        "rankOldAndNew": "OLD",
        "movieCd": "20256283",
        "movieNm": "호퍼스",
        "openDt": "2026-03-04",
        "salesAmt": "2141337660",
        "salesShare": "8.7",
        "sale

In [4]:
# 원하는 데이터 추출

items_m = data_m["boxOfficeResult"]["weeklyBoxOfficeList"]
date1 = data_m['boxOfficeResult']['showRange'][:8]
date2 = data_m['boxOfficeResult']['showRange'][9:]
stDate = datetime.strptime(date1, "%Y%m%d").strftime("%Y-%m-%d")
edDate = datetime.strptime(date2, "%Y%m%d").strftime("%Y-%m-%d")

print(f"<{stDate} ~ {edDate} {data_m['boxOfficeResult']['boxofficeType']}>")
print(' | 순위 | 영화 제목 | 개봉일 | 누적 관객수 |\n')
for item in items_m:
    print(f" 🎬 {item['rank']}위 {item['movieNm']} | {item['openDt']} | {int(item['audiAcc']):,}명")

<2026-03-09 ~ 2026-03-15 주간 박스오피스>
 | 순위 | 영화 제목 | 개봉일 | 누적 관객수 |

 🎬 1위 왕과 사는 남자 | 2026-02-04 | 13,467,690명
 🎬 2위 호퍼스 | 2026-03-04 | 531,160명
 🎬 3위 삼악도 | 2026-03-11 | 57,385명
 🎬 4위 F1 더 무비 | 2025-06-25 | 5,261,209명
 🎬 5위 휴민트 | 2026-02-11 | 1,967,601명
 🎬 6위 매드 댄스 오피스 | 2026-03-04 | 43,310명
 🎬 7위 오만과 편견 | 2006-03-24 | 898,112명
 🎬 8위 신의악단 | 2025-12-31 | 1,432,173명
 🎬 9위 초속 5센티미터 | 2026-02-25 | 90,873명
 🎬 10위 극장판 진격의 거인 완결편 더 라스트 어택 | 2025-03-13 | 960,945명


In [5]:
my_params2 = {
    "key": API_KEY_M,
    "targetDt": "20260525",
    "itemPerPage": "10",
    "weekGb": "0", # 주간:0 / 주말:1 / 주중:2
    "repNationCd": "K" # 한국:K / 외국:F
}

if my_url:
    my_response2 = requests.get(my_url, params=my_params2)
    print("응답 코드:", my_response2.status_code)
    if my_response2.status_code == 200:
        data_m2 = my_response2.json()
        pretty(data_m2)

응답 코드: 200
{
  "boxOfficeResult": {
    "boxofficeType": "주간 박스오피스",
    "showRange": "20260525~20260531",
    "yearWeekTime": "202622",
    "weeklyBoxOfficeList": [
      {
        "rnum": "1",
        "rank": "1",
        "rankInten": "0",
        "rankOldAndNew": "OLD",
        "movieCd": "20252402",
        "movieNm": "군체",
        "openDt": "2026-05-21",
        "salesAmt": "20803427660",
        "salesShare": "94.7",
        "salesInten": "4158910210",
        "salesChange": "25.0",
        "salesAcc": "37451755110",
        "audiCnt": "1973299",
        "audiInten": "472116",
        "audiChange": "31.4",
        "audiAcc": "3474888",
        "scrnCnt": "2060",
        "showCnt": "54580"
      },
      {
        "rnum": "2",
        "rank": "2",
        "rankInten": "17",
        "rankOldAndNew": "OLD",
        "movieCd": "20263512",
        "movieNm": "이상한 과자 가게 전천당",
        "openDt": "2026-05-29",
        "salesAmt": "558647650",
        "salesShare": "2.5",
        "salesInt

In [6]:
items_m2 = data_m2["boxOfficeResult"]["weeklyBoxOfficeList"]
stDate2 = datetime.strptime(data_m2['boxOfficeResult']['showRange'][:8], "%Y%m%d").strftime("%Y-%m-%d")
edDate2 = datetime.strptime(data_m2['boxOfficeResult']['showRange'][9:], "%Y%m%d").strftime("%Y-%m-%d")

print(f"<{stDate2} ~ {edDate2} 한국 영화 {data_m2['boxOfficeResult']['boxofficeType']}>")
print(' | 순위 | 영화 제목 | 개봉일 | 누적 관객수 |\n')
for item in items_m2:
    print(f" 🎬 {item['rank']}위 {item['movieNm']} | {item['openDt']} | {int(item['audiAcc']):,}명")

<2026-05-25 ~ 2026-05-31 한국 영화 주간 박스오피스>
 | 순위 | 영화 제목 | 개봉일 | 누적 관객수 |

 🎬 1위 군체 | 2026-05-21 | 3,474,888명
 🎬 2위 이상한 과자 가게 전천당 | 2026-05-29 | 60,421명
 🎬 3위 와일드 씽 | 2026-06-03 | 19,981명
 🎬 4위 살목지 | 2026-04-08 | 3,238,030명
 🎬 5위 왕과 사는 남자 | 2026-02-04 | 16,889,051명
 🎬 6위 쇼타씨의 마지막 출장 | 2026-05-27 | 5,752명
 🎬 7위 교생실습 | 2026-05-13 | 33,502명
 🎬 8위 내 이름은 | 2026-04-15 | 217,381명
 🎬 9위 남태령 | 2026-05-20 | 5,194명
 🎬 10위 란 12.3 | 2026-04-22 | 247,797명


---
# 📊 심화: pandas로 데이터 정리하기

위 실습에서 가져온 데이터를 DataFrame으로 만들어 분석해봅시다!

In [12]:
movie_data = [] # 전체 영화 정보

for i in items_m:
    movie_data_dict = {}
    rank = i['rank'] # 순위
    title = i['movieNm'] # 영화 제목
    openDate = i['openDt'] # 개봉일
    count = i['audiAcc'] # 누적 관객수
    old_new = i['rankOldAndNew'] # 신규 진입 여부

    movie_data_dict['순위'] = rank
    movie_data_dict['제목'] = title
    movie_data_dict['개봉일'] = openDate
    movie_data_dict['누적 관객수'] = count
    movie_data_dict['신규 진입 여부'] = old_new

    movie_data.append(movie_data_dict)

movie_data

[{'순위': '1',
  '제목': '왕과 사는 남자',
  '개봉일': '2026-02-04',
  '누적 관객수': '13467690',
  '신규 진입 여부': 'OLD'},
 {'순위': '2',
  '제목': '호퍼스',
  '개봉일': '2026-03-04',
  '누적 관객수': '531160',
  '신규 진입 여부': 'OLD'},
 {'순위': '3',
  '제목': '삼악도',
  '개봉일': '2026-03-11',
  '누적 관객수': '57385',
  '신규 진입 여부': 'OLD'},
 {'순위': '4',
  '제목': 'F1 더 무비',
  '개봉일': '2025-06-25',
  '누적 관객수': '5261209',
  '신규 진입 여부': 'NEW'},
 {'순위': '5',
  '제목': '휴민트',
  '개봉일': '2026-02-11',
  '누적 관객수': '1967601',
  '신규 진입 여부': 'OLD'},
 {'순위': '6',
  '제목': '매드 댄스 오피스',
  '개봉일': '2026-03-04',
  '누적 관객수': '43310',
  '신규 진입 여부': 'OLD'},
 {'순위': '7',
  '제목': '오만과 편견',
  '개봉일': '2006-03-24',
  '누적 관객수': '898112',
  '신규 진입 여부': 'NEW'},
 {'순위': '8',
  '제목': '신의악단',
  '개봉일': '2025-12-31',
  '누적 관객수': '1432173',
  '신규 진입 여부': 'OLD'},
 {'순위': '9',
  '제목': '초속 5센티미터',
  '개봉일': '2026-02-25',
  '누적 관객수': '90873',
  '신규 진입 여부': 'OLD'},
 {'순위': '10',
  '제목': '극장판 진격의 거인 완결편 더 라스트 어택',
  '개봉일': '2025-03-13',
  '누적 관객수': '960945',
  '신규 진입 여부': 'NEW'}]

In [13]:
df = pd.DataFrame(movie_data).set_index('순위')
df

,제목,개봉일,누적 관객수,신규 진입 여부
순위,,,,
1,왕과 사는 남자,2026-02-04,13467690,OLD
2,호퍼스,2026-03-04,531160,OLD
3,삼악도,2026-03-11,57385,OLD
4,F1 더 무비,2025-06-25,5261209,NEW
5,휴민트,2026-02-11,1967601,OLD
6,매드 댄스 오피스,2026-03-04,43310,OLD
7,오만과 편견,2006-03-24,898112,NEW
8,신의악단,2025-12-31,1432173,OLD
9,초속 5센티미터,2026-02-25,90873,OLD


In [10]:
df2 = pd.DataFrame(items_m2)
col_name = {
    'rank':'순위',
    'rankOldAndNew':'신규 진입 여부',
    'movieNm':'제목',
    'openDt':'개봉일',
    'audiAcc':'누적 관객수'
}
df2 = df2.rename(columns=col_name)
df2_view = df2[['순위', '제목', '개봉일', '누적 관객수', '신규 진입 여부']].set_index('순위')
df2_view

,제목,개봉일,누적 관객수,신규 진입 여부
순위,,,,
1,군체,2026-05-21,3474888,OLD
2,이상한 과자 가게 전천당,2026-05-29,60421,OLD
3,와일드 씽,2026-06-03,19981,OLD
4,살목지,2026-04-08,3238030,OLD
5,왕과 사는 남자,2026-02-04,16889051,OLD
6,쇼타씨의 마지막 출장,2026-05-27,5752,OLD
7,교생실습,2026-05-13,33502,OLD
8,내 이름은,2026-04-15,217381,OLD
9,남태령,2026-05-20,5194,OLD


In [ ]:
# 💾 CSV로 저장 (엑셀에서도 열 수 있음)
# df_food_kr.to_csv('대구_중구_맛집.csv', index=False, encoding='utf-8-sig')  # utf-8-sig: 엑셀 한글 깨짐 방지
# print('✅ 대구_중구_맛집.csv 저장 완료!')
# print(f'   → {len(df_food_kr)}행 × {len(df_food_kr.columns)}열')

# 저장된 파일 확인
# df_check = pd.read_csv('대구_중구_맛집.csv')
# df_check.head(3)

---
# 📝 오늘 배운 것 정리

| 실습 | API | 새로 배운 것 | 어려웠던 점 |
|------|-----|------------|------------|
| 1. 날씨 | 기상청 | | |
| 2. 음식점 | 지자체 | | |
| 3. 병원/약국 | 심평원 | | |
| 4. 관광/행사 | 관광공사 | | |
| 5. 주차장 | (직접 찾기) | | |
| 6. 의약품 | 식약처 | | |
| 자유 | (직접 찾기) | | |

---
### 💡 핵심 정리
```python
# 모든 공공데이터 API의 공통 패턴
import requests

API_KEY = "본인키".strip()           # 1. 키 (앞뒤 공백 제거 필수!)
url = "https://apis.data.go.kr/..." # 2. URL (API 문서에서 확인)
params = {                          # 3. 파라미터 (필수값 확인)
    "serviceKey": API_KEY,
    "numOfRows": "10",
    "pageNo": "1",
    "_type": "json"
}
res = requests.get(url, params=params)  # 4. 요청
data = res.json()                       # 5. JSON 변환
items = data["response"]["body"]["items"]["item"]  # 6. 데이터 추출
df = pd.DataFrame(items)                # 7. DataFrame 변환
```

### 다음 시간 🔜
- folium으로 지도에 시각화하기
- matplotlib/plotly로 차트 그리기  
- 여러 API 데이터 합쳐서 분석하기